# 14 — RF, ExtraTrees và HistGradientBoosting
Bằng chứng thực thi cho ba baseline bảng trên cùng full-data split, preprocessor và năm seed. Sau khi chạy xong, hãy lưu notebook với toàn bộ output.

In [ ]:
from pathlib import Path
import json, os, subprocess
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'research/server').is_dir():
    raise RuntimeError('Hãy mở Jupyter tại thư mục gốc của repo')
def stage(name):
    subprocess.run(['bash', 'research/server/run_next_experiments.sh', name], cwd=ROOT, check=True)

## Huấn luyện 120 baseline run
4 dataset × 2 task × 3 thuật toán × 5 seed. Cell hỗ trợ resume an toàn.

In [ ]:
stage('tabular_train')

## Replay độc lập
Nạp lại từng model, dự đoán toàn bộ test, đối chiếu metric, probability và hash artefact.

In [ ]:
stage('tabular_verify')

In [ ]:
root = ROOT / 'research/artifacts/tabular_full_runs'
runs = pd.read_csv(root / 'runs.csv')
verification = json.loads((ROOT / 'research/results/tabular_full_verification.json').read_text())
expected_models = {'random_forest', 'extra_trees', 'hist_gradient_boosting'}
assert len(runs) == 120
assert set(runs.model) == expected_models
assert set(runs.seed) == {11, 22, 33, 44, 55}
assert verification['passed'] is True and verification['runs_checked'] == 120
display(runs.groupby(['dataset', 'task', 'model']).test_macro_f1.agg(['count', 'mean', 'std', 'min', 'max']))
print({k: verification[k] for k in ('passed', 'runs_checked', 'largest_metric_recomputation_error', 'largest_probability_replay_error')})